In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import xarray as xr
import matplotlib.pyplot as plt
import tqdm as tqdm


In [2]:
SM_data_all = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_7-10-2024.txt')

In [29]:
#import missing data files
cmi_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_CMI.txt', delimiter='\t')
mon_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_MON.txt', delimiter='\t')
siu_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_SIU.txt', delimiter='\t')
#concatenate all data
SM_data_all = pd.concat([SM_data_all, cmi_data, mon_data, siu_data], axis = 0)
SM_data_all = SM_data_all.reset_index(drop = True)


In [35]:
SM_data_all

,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv),Precip (in)
0,bbc,2010-07-16 12:00:00.000,0.0,0.237,0.270,0.154,0.167,0.164,0.077,NaN
1,bbc,2010-07-16 13:00:00.000,0.0,0.236,0.266,0.152,0.168,0.164,0.077,NaN
2,bbc,2010-07-16 14:00:00.000,0.0,0.235,0.266,0.152,0.167,0.162,0.079,NaN
3,bbc,2010-07-16 15:00:00.000,0.0,0.234,0.265,0.152,0.166,0.163,0.078,NaN
4,bbc,2010-07-16 16:00:00.000,0.0,0.233,0.265,0.152,0.168,0.163,0.077,NaN
...,...,...,...,...,...,...,...,...,...,...
2714121,siu,12/31/23 20:00,NaN,0.317,0.362,0.322,0.298,0.298,0.461,0.0
2714122,siu,12/31/23 21:00,NaN,0.318,0.362,0.323,0.299,0.297,0.461,0.0
2714123,siu,12/31/23 22:00,NaN,0.317,0.361,0.323,0.300,0.295,0.461,0.0
2714124,siu,12/31/23 23:00,NaN,0.317,0.361,0.323,0.298,0.296,0.461,0.0


What is wfv?
water fraction volume cm3/cm3 volumetric fraction of soil column that is water (relative to total 
absolut volume)

In [4]:
Stations = SM_data_all.Station.unique()

In [5]:
#clean data, remove impossible values
def clean_data(data):
    Data_column_names = ['Precipitation (in)', 'Soil Moisture 5cm (wfv)', 'Soil Moisture 10cm (wfv)', 
                     'Soil Moisture 20cm (wfv)','Soil Moisture 50cm (wfv)', 
                     'Soil Moisture 100cm (wfv)', 'Soil Moisture 150cm (wfv)']
    for i in range(len(Data_column_names)):
        data.loc[data[Data_column_names[i]] > 1000, Data_column_names[i]] = pd.NA

    #station's data are not linear, but scrabmled, put in order
    Stations = data.Station.unique()
    for s in Stations:
        station_data = data.loc[data['Station'] == s]
        station_data = station_data.sort_values(by = 'Date & Time (CST)')
        station_data = station_data.reset_index(drop = True)
        if s == 'bbc':
            new_df = station_data
        else:
            new_df = pd.concat([new_df, station_data], axis = 0)

    new_df = new_df.reset_index(drop = True)


    return new_df

SM_data_cleaned= clean_data(SM_data_all)

In [6]:
SM_data_cleaned

,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv)
0,bbc,2008-01-01 01:00:00,4.13,0.270,0.280,0.200,0.200,0.180,0.060
1,bbc,2008-01-01 02:00:00,4.13,0.270,0.280,0.190,0.200,0.180,0.050
2,bbc,2008-01-01 03:00:00,4.13,0.270,0.280,0.200,0.200,0.180,0.060
3,bbc,2008-01-01 04:00:00,4.13,0.270,0.280,0.190,0.210,0.180,0.060
4,bbc,2008-01-01 05:00:00,4.13,0.270,0.280,0.200,0.210,0.180,0.060
...,...,...,...,...,...,...,...,...,...
2312329,ste,2023-12-31 20:00:00.000,0.00,0.362,0.384,0.322,0.371,0.447,0.737
2312330,ste,2023-12-31 21:00:00.000,0.00,0.362,0.384,0.321,0.371,0.447,0.737
2312331,ste,2023-12-31 22:00:00.000,0.00,0.362,0.384,0.321,0.371,0.447,0.737
2312332,ste,2023-12-31 23:00:00.000,0.00,0.363,0.384,0.322,0.371,0.447,0.737


In [7]:
Column_sum_depth = [10, 100]
sensor_depths = np.array([5, 10, 20, 50, 100, 150])

#Column_depth is Column_sum_depth, and depths is sensor_depths
def column_weighted_sm(data, Column_depth, depths):
    for m in range(len(Column_depth)):
        n = np.where(sensor_depths == Column_depth[m])[0][0]
        depth_weights = np.zeros(len(depths))
        Weighted_column_names = str(Column_depth[m]) + 'cm_weighted_col_sm'

        for r in range(len(sensor_depths[0:n]) + 1):
            if r == 0:
                depth_weights[r] = (((sensor_depths[r+1] - sensor_depths[r])/2 + 
                                     sensor_depths[r])/Column_depth[m])
            elif ((r >= 0) and (r < len(sensor_depths[0:n])-1)):
                depth_weights[r] = ((sensor_depths[r+1] - sensor_depths[r-1])/2)/Column_depth[m] 
            else:
                depth_weights[r] = ((sensor_depths[r] - sensor_depths[r-1])/2)/Column_depth[m]
        print('depth weights =', depth_weights)

        for c in range(len(sensor_depths[0:n]) + 1):
            if c == 0:
                data[Weighted_column_names] = depth_weights[c] * data.iloc[:,c + 3]
            else:
                data[Weighted_column_names] =( data[Weighted_column_names] +
                                              depth_weights[c] * data.iloc[:,c + 3])

        #lagged_weighted_sm = str(Column_depth[m]) + 'cm_weighted_col_sm_lagged'
        #data[lagged_weighted_sm] = data[Weighted_column_names].shift(1)
        #dsm = str(Column_depth[m]) + 'cm_weighted_col_sm_delta'
        #data[dsm] = data[Weighted_column_names].diff()

    return data

SM_data_weighted = column_weighted_sm(SM_data_cleaned, Column_sum_depth, sensor_depths)
SM_data_weighted.head()    

depth weights = [0.75 0.25 0.   0.   0.   0.  ]
depth weights = [0.075 0.075 0.2   0.15  0.25  0.   ]


,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv),10cm_weighted_col_sm,100cm_weighted_col_sm
0,bbc,2008-01-01 01:00:00,4.13,0.27,0.28,0.20,0.20,0.18,0.06,0.2725,0.15625
1,bbc,2008-01-01 02:00:00,4.13,0.27,0.28,0.19,0.20,0.18,0.05,0.2725,0.15425
2,bbc,2008-01-01 03:00:00,4.13,0.27,0.28,0.20,0.20,0.18,0.06,0.2725,0.15625
3,bbc,2008-01-01 04:00:00,4.13,0.27,0.28,0.19,0.21,0.18,0.06,0.2725,0.15575
4,bbc,2008-01-01 05:00:00,4.13,0.27,0.28,0.20,0.21,0.18,0.06,0.2725,0.15775


Missing some stations, no champaign station cci

## Partition storm events: 0 rain for 24 consecutive hours (eventually switch to 20/24 maybe)

ID indices for beginning and end of dry periods, subtract the difference and locate the dry periods 
greater than 24 hours and set those to be the cut off of the storm events

In [14]:
consecutive_dry_hours = 24
def precip_event_ID(data, hours):
    #min_num_dry_hours = np.floor(hours * 0.843) #(20/24)=0.8333333 same for 30/36
    Stations = data.Station.unique()
    data['Rain_ID'] = (data['Precipitation (in)'] > 0).astype(int)
    for s in Stations:
        beginning_of_station = data.loc[data['Station'] == s].index[0]
        data.iloc[beginning_of_station, 11] = np.NaN
        
    data['Rain_ID_lag'] = data['Rain_ID'].shift(1)
    data['Rain_ID_diff'] = data['Rain_ID'].sub(data['Rain_ID_lag'], axis = 0) #values of -1 in this column will always represent when a dry period started

    data['Dry_periods'] = data['Rain_ID_diff']
    data['Dry_periods'] = data['Dry_periods'].replace(1, -1) #now the days when dry periods start and end are 
    #represented as -1. Use index to grab the indices of these dates and subtract
    
    data['Dry_periods_overmin'] = False #set to false, find indices over 24 from dry period indices 
    #spacing and replace those with TRUE then select those with 
    print(Stations)
    for sn in Stations:
        station_data = data.loc[data['Station'] == sn]
        dry_period_indices = pd.Series(station_data.loc[station_data['Dry_periods'] == -1].index)
        dry_period_indices_spacing = dry_period_indices.diff()
        dry_period_indices_spacing_shifted = dry_period_indices_spacing.shift(-1) #aligning period lengths with the index they start with 
        dry_over_consec_min = (dry_period_indices_spacing_shifted >= hours)
        dry_periods = pd.concat([dry_period_indices, dry_period_indices_spacing_shifted], axis = 1)
        dry_periods = dry_periods.loc[dry_over_consec_min]
        
        data.iloc[dry_periods[0], 15] = True
        beginning_of_station = data.loc[data['Station'] == sn].index[0]
        #assign last portion of data for each station that is less than consecutive dry hours to be 
        #False as we cannot no for certain if it meets the dry period cutoff definition
        data.iloc[beginning_of_station-consecutive_dry_hours:beginning_of_station, 15] = False


    return data

WeightedSM_data_stormID = precip_event_ID(SM_data_weighted, consecutive_dry_hours)

['bbc' 'brw' 'bvl' 'dek' 'dxs' 'fai' 'fre' 'frm' 'icc' 'llc' 'oln' 'orr'
 'rnd' 'sfm' 'sni' 'stc' 'ste']


In [16]:
Stations = WeightedSM_data_stormID.Station.unique()
for s in Stations:
    beginning_of_station = WeightedSM_data_stormID.loc[WeightedSM_data_stormID['Station'] == s].index[0]
    print(WeightedSM_data_stormID.iloc[beginning_of_station - 10 : beginning_of_station + 10, 11:16])

Empty DataFrame
Columns: [Rain_ID, Rain_ID_lag, Rain_ID_diff, Dry_periods, Dry_periods_overmin]
Index: []
        Rain_ID  Rain_ID_lag  Rain_ID_diff  Dry_periods  Dry_periods_overmin
152957      0.0          0.0           0.0          0.0                False
152958      0.0          0.0           0.0          0.0                False
152959      0.0          0.0           0.0          0.0                False
152960      0.0          0.0           0.0          0.0                False
152961      0.0          0.0           0.0          0.0                False
152962      0.0          0.0           0.0          0.0                False
152963      0.0          0.0           0.0          0.0                False
152964      0.0          0.0           0.0          0.0                False
152965      0.0          0.0           0.0          0.0                False
152966      0.0          0.0           0.0          0.0                False
152967      NaN          0.0           NaN     

In [13]:
WeightedSM_data_stormID

,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv),10cm_weighted_col_sm,100cm_weighted_col_sm,Rain_ID,Rain_ID_lag,Rain_ID_diff,Dry_periods,Dry_periods_overmin
0,bbc,2008-01-01 01:00:00,4.13,0.270,0.280,0.200,0.200,0.180,0.060,0.27250,0.156250,NaN,NaN,NaN,NaN,False
1,bbc,2008-01-01 02:00:00,4.13,0.270,0.280,0.190,0.200,0.180,0.050,0.27250,0.154250,1.0,NaN,NaN,NaN,False
2,bbc,2008-01-01 03:00:00,4.13,0.270,0.280,0.200,0.200,0.180,0.060,0.27250,0.156250,1.0,1.0,0.0,0.0,False
3,bbc,2008-01-01 04:00:00,4.13,0.270,0.280,0.190,0.210,0.180,0.060,0.27250,0.155750,1.0,1.0,0.0,0.0,False
4,bbc,2008-01-01 05:00:00,4.13,0.270,0.280,0.200,0.210,0.180,0.060,0.27250,0.157750,1.0,1.0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2312329,ste,2023-12-31 20:00:00.000,0.00,0.362,0.384,0.322,0.371,0.447,0.737,0.36750,0.287750,0.0,0.0,0.0,0.0,False
2312330,ste,2023-12-31 21:00:00.000,0.00,0.362,0.384,0.321,0.371,0.447,0.737,0.36750,0.287550,0.0,0.0,0.0,0.0,False
2312331,ste,2023-12-31 22:00:00.000,0.00,0.362,0.384,0.321,0.371,0.447,0.737,0.36750,0.287550,0.0,0.0,0.0,0.0,False
2312332,ste,2023-12-31 23:00:00.000,0.00,0.363,0.384,0.322,0.371,0.447,0.737,0.36825,0.287825,0.0,0.0,0.0,0.0,False


### select rows where Rain_ID_diff = -1 (beginning of dry period) and Dry_periods_over24 = True this is where the storm event ends
### Rain_ID_diff = 1 is beginning
select sm for timestep before beginning

In [12]:
def Rain_events_data(data):
    event_beginning_indices = pd.Series(data.loc[data['Rain_ID_diff'] == 1].index)
    event_ending_indices = pd.Series(data.loc[(data['Rain_ID_diff'] == -1) & 
                                                (data['Dry_periods_overmin'] == True)].index)
    
    event_df = pd.concat([event_beginning_indices, event_ending_indices], axis = 1)
    print(event_df)

    return new_df

test = Rain_events_data(WeightedSM_data_stormID)

             0       1
0         1042   230.0
1         2673  2367.0
2         4115  3704.0
3         6122  6059.0
4         6132  6179.0
...        ...     ...
51461  2312223     NaN
51462  2312245     NaN
51463  2312273     NaN
51464  2312319     NaN
51465  2312325     NaN

[51466 rows x 2 columns]


NameError: name 'new_df' is not defined